# DL

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.neural_network import MLPClassifier

# ==========================================
# 1. 데이터 불러오기 (트리 기반 모델용 unscaled)
# ==========================================
# XGBoost는 트리 기반 모델이므로 스케일링이 안 된 데이터를 부릅니다.
print("📥 데이터를 불러오고 K-Fold를 위해 병합합니다...")

# 기존에 쪼개져 있던 Train과 Val 데이터를 불러옵니다.
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')

y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

# K-Fold가 전체 데이터에서 자유롭게 쪼갤 수 있도록 하나로 합칩니다.
X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

print(f"✅ 병합 완료! (전체 학습 데이터 X: {X.shape}, y: {y.shape})")

# ==========================================
# 🌟 1.5 K-Fold 시작 전 Optuna 하이퍼파라미터 탐색 추가
# ==========================================
import optuna
from sklearn.model_selection import train_test_split

print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 8:2 분할)...")
# 전체 X, y를 Optuna가 빨리 학습해볼 수 있게 임시로 분할합니다.
X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_weight_opt = (y_opt_tr == 0).sum() / (y_opt_tr == 1).sum()

def objective(trial):
    params = {
        # 은닉층(Hidden Layer)의 크기와 개수를 컴퓨터가 조합해 봅니다.
        'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', [(64, 32), (128, 64, 32), (64,)]),
        # 과적합 방지를 위한 L2 규제 (XGB의 reg_lambda와 비슷)
        'alpha': trial.suggest_float('alpha', 0.0001, 0.1, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 0.001, 0.05, log=True),
        'max_iter': 1000,          # 충분한 반복 횟수
        'early_stopping': True,    # 신경망 자체적인 조기 종료 켜기
        'random_state': 42
    }
    
    model_opt = MLPClassifier(**params)
    
    # 🚨 신경망도 내부적으로 분할하므로 여기서 eval_set은 넣지 않습니다.
    model_opt.fit(X_opt_tr, y_opt_tr)
    
    return roc_auc_score(y_opt_val, model_opt.predict_proba(X_opt_val)[:, 1])

# 30회 탐색 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print(f"🎉 탐색 완료! 최고 Validation AUC: {study.best_value:.4f}")

# ==========================================
# 2. K-Fold 설정 및 XGBoost 모델 학습
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))          # 5번 검증한 결과를 모아둘 빈 배열
test_pred = np.zeros(len(X_test))    # 5번 예측한 테스트 결과를 누적할 빈 배열
fold_scores = []

print("\n🚀 XGBoost K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # 기존에 세팅하셨던 모델 파라미터 그대로 적용
    # Fold별 클래스 불균형 비율 계산
    scale_pos_weight_fold = (y_tr == 0).sum() / (y_tr == 1).sum()

# 🌟 Optuna 결과 덮어쓰기
    model = MLPClassifier(
        **study.best_params,
        max_iter=1000,
        early_stopping=True,
        random_state=42 + fold
    )
    
    # 🚨 eval_set 없음
    model.fit(X_tr, y_tr)
    # 검증 데이터 예측 및 OOF 저장
    val_proba = model.predict_proba(X_va)[:, 1]
    oof_pred[val_idx] = val_proba
    
    # 테스트 데이터 예측 (5로 나누어 누적 합산 -> 최종적으로 평균이 됨)
    test_pred += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    # Fold별 점수 기록
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
# 기본 임계값 0.5를 기준으로 F1 및 정확도 계산
oof_label = (oof_pred >= 0.5).astype(int)
overall_f1 = f1_score(y, oof_label)
overall_acc = accuracy_score(y, oof_label)

print("\n--- 📊 최종 Validation(OOF) 결과 ---")
print(f"정확도 (Accuracy): {overall_acc:.4f}")
print(f"F1 점수 (F1 Score): {overall_f1:.4f}")
print(f"AUC (ROC-AUC):  {overall_auc:.4f}")
print(f"Fold별 AUC: {[round(s, 4) for s in fold_scores]}")


## Main

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

print("📥 PyTorch 딥러닝을 위한 스케일링 데이터를 불러옵니다...")

# 🚨 딥러닝은 반드시 스케일링된 데이터를 사용해야 합니다.
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')
y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

# 테스트 데이터를 미리 텐서(딥러닝 전용 데이터 타입)로 변환해 둡니다.
X_test_tensor = torch.FloatTensor(X_test.values)

# ==========================================
# 🧩 부품 1: 신경망 아키텍처 (도면 설계)
# ==========================================
class ReturnRiskNet(nn.Module):
    def __init__(self, input_dim):
        super(ReturnRiskNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),   # 학습을 안정적으로 만들어주는 부품
            nn.ReLU(),
            nn.Dropout(0.3),      # 과적합 방지
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(32, 1),
            nn.Sigmoid()          # 0~1 사이 확률값 출력
        )

    def forward(self, x):
        return self.network(x)

# ==========================================
# 2. K-Fold 설정 및 PyTorch 학습 루프 시작
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_scores = []

# 학습용 파라미터 (원래라면 Optuna로 찾았을 값들을 고정값으로 설정)
EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 0.005

print("\n🚀 PyTorch K-Fold 딥러닝 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- Fold {fold} ---")
    
    # 1) Fold별 데이터 분할
    X_tr, X_va = X.iloc[train_idx].values, X.iloc[val_idx].values
    y_tr, y_va = y.iloc[train_idx].values, y.iloc[val_idx].values
    
    # 2) 🧩 부품 2: 데이터 로더 (텐서로 변환하고 식판에 담기)
    # y값의 형태를 (N,) 에서 (N, 1) 로 세워주어야 딥러닝이 인식합니다.
    train_dataset = TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr).unsqueeze(1))
    val_dataset = TensorDataset(torch.FloatTensor(X_va), torch.FloatTensor(y_va).unsqueeze(1))
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 3) 모델, 손실 함수, 최적화 도구 세팅
    model = ReturnRiskNet(input_dim=X.shape[1])
    criterion = nn.BCELoss() # 이진 분류용 손실 함수
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # 4) 🧩 부품 3: 학습 루프 (수동 변속 과정)
    for epoch in range(EPOCHS):
        model.train() # 학습 모드 ON
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()            # 1. 기울기 초기화
            predictions = model(batch_X)     # 2. 예측
            loss = criterion(predictions, batch_y) # 3. 오차 계산
            loss.backward()                  # 4. 역전파 (어디 고칠지 계산)
            optimizer.step()                 # 5. 가중치 업데이트
            
    # 5) 검증 데이터 예측 및 OOF 저장
    model.eval() # 평가 모드 ON (Dropout 등 정지)
    with torch.no_grad(): # 예측 시에는 기울기 계산을 끔(메모리 절약)
        val_proba = model(torch.FloatTensor(X_va)).squeeze().numpy()
        oof_pred[val_idx] = val_proba
        
        # 테스트 데이터 예측 (5로 나누어 누적)
        test_proba = model(X_test_tensor).squeeze().numpy()
        test_pred += test_proba / skf.n_splits
        
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# ==========================================
# 3. 전체 OOF 성능 평가
# ==========================================
overall_auc = roc_auc_score(y, oof_pred)
oof_label = (oof_pred >= 0.5).astype(int)
print(f"\n🏆 최종 PyTorch OOF ROC-AUC: {overall_auc:.4f}")



In [ ]:
# ==========================================
# 4. 대시보드 연동용 CSV 일괄 추출 (PyTorch용)
# ==========================================
model_name = "PyTorch" 
print("\n💾 대시보드 및 제출용 CSV 파일 추출을 시작합니다...")

# 1. Validation 예측 결과
val_df = pd.DataFrame({'pred_prob': oof_pred, 'pred_label': oof_label, 'returned': y.values})
val_df.to_csv('val_predictions.csv', index=False)

# 2. Test 예측 결과 및 제출용
test_label = (test_pred >= 0.5).astype(int)
test_df = pd.DataFrame({'order_id': test_id_df['order_id'], 'pred_prob': test_pred, 'pred_label': test_label})
test_df.to_csv('test_predictions.csv', index=False)
test_df[['order_id', 'pred_prob']].rename(columns={'pred_prob': 'returned'}).to_csv(f'{model_name}_submission.csv', index=False)

print("✅ [1/2] 예측 결과 및 제출용 파일 생성 완료!")
print("⚠️ [2/2] PyTorch 신경망은 자체적인 변수 중요도를 제공하지 않아 생략합니다.")

# 대시보드 에러 방지 백그라운드 처리
X.to_csv('X_val_unscaled.csv', index=False)
pd.DataFrame({'returned': y}).to_csv('y_val.csv', index=False)
print("\n🎉 파이토치 모델 추출 완료!")

In [ ]:
# ! pip install pytorch
! pip install torch

In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import optuna

print("📥 PyTorch 딥러닝을 위한 스케일링 데이터를 불러옵니다...")

# 🚨 딥러닝 전용 데이터 (Scaled)
X_train_part = pd.read_csv('X_tr_scaled.csv')
X_val_part = pd.read_csv('X_val_scaled.csv')
y_train_part = pd.read_csv('y_tr.csv')['returned']
y_val_part = pd.read_csv('y_val.csv')['returned']

X = pd.concat([X_train_part, X_val_part], axis=0).reset_index(drop=True)
y = pd.concat([y_train_part, y_val_part], axis=0).reset_index(drop=True)

X_test = pd.read_csv('X_test_scaled.csv')
test_id_df = pd.read_csv('test_order_id.csv')

X_test_tensor = torch.FloatTensor(X_test.values)

# ==========================================
# 🧩 부품 1: 유연한 신경망 아키텍처 (Optuna가 조립할 수 있게 설계)
# ==========================================
class DynamicReturnNet(nn.Module):
    def __init__(self, input_dim, h1, h2, drop_rate):
        super(DynamicReturnNet, self).__init__()
        # Optuna가 던져주는 h1(1층 두께), h2(2층 두께), drop_rate(규제)에 따라 모양이 바뀝니다.
        self.network = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            
            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            
            nn.Linear(h2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

# ==========================================
# 🌟 2. Optuna 하이퍼파라미터 탐색 (임시 8:2 분할로 빠르게)
# ==========================================
print("\n🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 분할 사용)...")

X_opt_tr, X_opt_val, y_opt_tr, y_opt_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 텐서 및 식판(DataLoader) 준비
opt_train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_opt_tr.values), torch.FloatTensor(y_opt_tr.values).unsqueeze(1)),
    batch_size=256, shuffle=True
)
X_opt_val_tensor = torch.FloatTensor(X_opt_val.values)

def objective(trial):
    # 1. 탐색할 파라미터 (층의 두께, 드롭아웃 비율, 학습률)
    h1 = trial.suggest_categorical('h1', [64, 128, 256])
    h2 = trial.suggest_categorical('h2', [16, 32, 64])
    drop_rate = trial.suggest_float('drop_rate', 0.1, 0.5)
    lr = trial.suggest_float('lr', 0.001, 0.01, log=True)
    
    # 2. 모델 및 학습 도구 조립
    model_opt = DynamicReturnNet(X.shape[1], h1, h2, drop_rate)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model_opt.parameters(), lr=lr)
    
    # 3. 짧게 학습 (Optuna 탐색용이므로 15 에폭만 돌립니다)
    model_opt.train()
    for epoch in range(15):
        for batch_X, batch_y in opt_train_loader:
            optimizer.zero_grad()
            loss = criterion(model_opt(batch_X), batch_y)
            loss.backward()
            optimizer.step()
            
    # 4. 검증 점수(AUC) 계산
    model_opt.eval()
    with torch.no_grad():
        val_preds = model_opt(X_opt_val_tensor).squeeze().numpy()
        
    return roc_auc_score(y_opt_val, val_preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # 20번 탐색
print(f"\n🎉 탐색 완료! 최적의 파라미터: {study.best_params}")

# ==========================================
# 3. K-Fold 설정 및 PyTorch 최종 학습 (최적 파라미터 적용)
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_scores = []

# Optuna가 찾은 최고 설정값 가져오기
best_h1 = study.best_params['h1']
best_h2 = study.best_params['h2']
best_drop = study.best_params['drop_rate']
best_lr = study.best_params['lr']

print("\n🚀 Optuna 최적 파라미터로 PyTorch K-Fold 학습을 시작합니다...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[train_idx].values, X.iloc[val_idx].values
    y_tr, y_va = y.iloc[train_idx].values, y.iloc[val_idx].values
    
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr).unsqueeze(1)),
        batch_size=256, shuffle=True
    )
    
    # Optuna가 찾은 도면대로 신경망 생성!
    model = DynamicReturnNet(X.shape[1], best_h1, best_h2, best_drop)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=best_lr)
    
    # 실전 K-Fold 학습 (충분히 30 에폭)
    model.train()
    for epoch in range(30):
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
            
    # 검증 및 테스트 예측
    model.eval()
    with torch.no_grad():
        val_proba = model(torch.FloatTensor(X_va)).squeeze().numpy()
        oof_pred[val_idx] = val_proba
        test_pred += model(X_test_tensor).squeeze().numpy() / skf.n_splits
        
    fold_auc = roc_auc_score(y_va, val_proba)
    fold_scores.append(fold_auc)
    print(f"   - Fold {fold} 완료 (ROC-AUC: {fold_auc:.4f})")

# OOF 전체 성능 평가
overall_auc = roc_auc_score(y, oof_pred)
print(f"\n🏆 최종 PyTorch OOF ROC-AUC: {overall_auc:.4f}")



📥 PyTorch 딥러닝을 위한 스케일링 데이터를 불러옵니다...

🔍 Optuna 탐색을 시작합니다 (탐색 속도를 위해 임시 분할 사용)...


ValueError: Found input variables with inconsistent numbers of samples: [200000, 360000]

In [ ]:
# ==========================================
# 4. 대시보드 연동용 CSV 일괄 추출 (PyTorch용)
# ==========================================
model_name = "PyTorch" 
print("\n💾 대시보드 및 제출용 CSV 파일 추출을 시작합니다...")

# 1. Validation 예측 결과
val_df = pd.DataFrame({'pred_prob': oof_pred, 'pred_label': (oof_pred >= 0.5).astype(int), 'returned': y.values})
val_df.to_csv('val_predictions.csv', index=False)

# 2. Test 예측 결과 및 제출용
test_df = pd.DataFrame({'order_id': test_id_df['order_id'], 'pred_prob': test_pred, 'pred_label': (test_pred >= 0.5).astype(int)})
test_df.to_csv('test_predictions.csv', index=False)
test_df[['order_id', 'pred_prob']].rename(columns={'pred_prob': 'returned'}).to_csv(f'{model_name}_submission.csv', index=False)

print("✅ [1/2] 예측 결과 및 제출용 파일 생성 완료!")
print("⚠️ [2/2] PyTorch 신경망은 자체적인 변수 중요도를 제공하지 않아 생략합니다.")

X.to_csv('X_val_unscaled.csv', index=False)
pd.DataFrame({'returned': y}).to_csv('y_val.csv', index=False)
print("\n🎉 모든 파이토치 모델 학습 및 추출이 완료되었습니다!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display, Markdown

# -----------------------------
# 1. 폰트 설정 (윈도우 맑은 고딕 기본 적용, Mac일 경우 'AppleGothic')
# -----------------------------
plt.rcParams["font.family"] = "Malgun Gothic" 
plt.rcParams["axes.unicode_minus"] = False

# -----------------------------
# 2. 파생 변수 및 메트릭 계산 함수
# -----------------------------
def decode_onehot(df, prefix):
    cols = [c for c in df.columns if c.startswith(prefix + "_")]
    if not cols:
        return pd.Series(["unknown"] * len(df), index=df.index)
    return df[cols].idxmax(axis=1).str.replace(prefix + "_", "", regex=False)

def add_derived_columns(df):
    df = df.copy()
    df["product_category_label"] = decode_onehot(df, "product_category")
    df["device_type_label"] = decode_onehot(df, "device_type")
    df["shipping_method_label"] = decode_onehot(df, "shipping_method")
    df["payment_method_label"] = decode_onehot(df, "payment_method")

    df["is_delayed"] = (df["delivery_delay_days"] > 0).astype(int)
    df["delay_days_int"] = df["delivery_delay_days"].astype(int)

    df["discount_bin"] = pd.cut(df["discount_percent"], bins=[-0.1, 10, 30, 50, 100], labels=["0-10", "10-30", "30-50", "50+"])
    df["delay_bin"] = pd.cut(df["delay_days_int"], bins=[-0.1, 1, 3, 100], labels=["0-1", "2-3", "4+"])
    df["past_return_bin"] = pd.cut(df["past_return_rate"], bins=[-0.01, 0.2, 0.5, 1.0], labels=["Low", "Medium", "High"])
    df["risk_level"] = pd.cut(df["pred_prob"], bins=[0.0, 0.3, 0.7, 1.0], labels=["Low", "Medium", "High"], include_lowest=True)
    return df

def calculate_metrics(df):
    if len(df) == 0: return 0, 0, 0, 0
    y_true, y_pred = df["returned"], df["pred_label"]
    accuracy = (y_true == y_pred).mean()
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, f1, precision, recall

print("✅ 기본 설정 및 함수 정의 완료!")

# -----------------------------
# 1. 만들어둔 CSV 파일 로드
# -----------------------------
try:
    X_val = pd.read_csv("X_val_unscaled.csv")
    y_val = pd.read_csv("y_val.csv")
    val_pred = pd.read_csv("val_predictions.csv")
    feature_importance = pd.read_csv("feature_importance.csv")
    
    # Test 예측 결과가 있으면 불러오고, 없으면 통과
    import os
    if os.path.exists("test_predictions.csv"):
        test_pred = pd.read_csv("test_predictions.csv")
    else:
        test_pred = None
        
    print("✅ 데이터 로드 성공!")
except Exception as e:
    print(f"❌ 데이터 로드 실패: {e}")

# 데이터 병합 및 파생변수 생성
val_df = X_val.copy()
val_df["returned"] = y_val["returned"].values
val_df["pred_prob"] = val_pred["pred_prob"].values
val_df["pred_label"] = val_pred["pred_label"].values
val_df = add_derived_columns(val_df)

# -----------------------------
# 2. 대시보드 필터 시뮬레이션 (원하는 값으로 바꿔서 테스트 가능!)
# -----------------------------
selected_category = "전체" # 예: "의류", "전자기기" 등으로 변경 가능
selected_shipping = "전체" 
selected_risk = "전체"     # "Low", "Medium", "High"

filtered_df = val_df.copy()
if selected_category != "전체":
    filtered_df = filtered_df[filtered_df["product_category_label"] == selected_category]
if selected_shipping != "전체":
    filtered_df = filtered_df[filtered_df["shipping_method_label"] == selected_shipping]
if selected_risk != "전체":
    filtered_df = filtered_df[filtered_df["risk_level"].astype(str) == selected_risk]

print(f"🔍 필터링된 분석 대상 주문 수: {len(filtered_df):,} 건")

f_accuracy, f_f1, f_precision, f_recall = calculate_metrics(filtered_df)

display(Markdown("### 🎯 핵심 지표 및 모델 성능"))
kpi_df = pd.DataFrame({
    "지표": ["분석 대상 주문 수", "실제 반품률", "평균 예측 확률", "고위험 주문 비율(0.7이상)"],
    "값": [
        f"{len(filtered_df):,} 건",
        f"{filtered_df['returned'].mean():.1%}" if len(filtered_df)>0 else "0%",
        f"{filtered_df['pred_prob'].mean():.1%}" if len(filtered_df)>0 else "0%",
        f"{(filtered_df['pred_prob'] >= 0.7).mean():.1%}" if len(filtered_df)>0 else "0%"
    ]
})

perf_df = pd.DataFrame({
    "성능 지표": ["Accuracy", "F1 Score", "Precision", "Recall"],
    "값": [f"{f_accuracy:.4f}", f"{f_f1:.4f}", f"{f_precision:.4f}", f"{f_recall:.4f}"]
})

display(pd.concat([kpi_df, perf_df], axis=1))

# 혼동 행렬 (Confusion Matrix)
display(Markdown("### 🧮 실제값 vs 예측값 혼동행렬"))
cm = pd.crosstab(filtered_df["returned"], filtered_df["pred_label"], rownames=["Actual"], colnames=["Predicted"])
display(cm)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("📊 예측 분포 및 실제 반품 패턴 분석", fontsize=16, fontweight='bold')

# 1. 예측 확률 분포
axes[0, 0].hist(filtered_df["pred_prob"], bins=20, color='skyblue', edgecolor='black')
axes[0, 0].axvline(0.7, color='red', linestyle="--", label='위험 임계값(0.7)')
axes[0, 0].set_title("예측 확률 분포")
axes[0, 0].legend()

# 2. 리스크 레벨 분포
risk_counts = filtered_df["risk_level"].value_counts().reindex(["Low", "Medium", "High"])
axes[0, 1].bar(risk_counts.index.astype(str), risk_counts.values, color=['green', 'orange', 'red'])
axes[0, 1].set_title("리스크 레벨 분포")

# 3. 할인율 구간별 실제 반품률
discount_rate = filtered_df.groupby("discount_bin", as_index=False, observed=False)["returned"].mean()
axes[1, 0].bar(discount_rate["discount_bin"].astype(str), discount_rate["returned"], color='purple')
axes[1, 0].set_title("할인율 구간별 실제 반품률")

# 4. 배송 지연일수별 실제 반품률
delay_rate = filtered_df.groupby("delay_days_int", as_index=False)["returned"].mean().sort_values("delay_days_int")
axes[1, 1].plot(delay_rate["delay_days_int"], delay_rate["returned"], marker="o", color='darkred')
axes[1, 1].set_title("배송 지연일수별 실제 반품률")

plt.tight_layout()
plt.show()

display(Markdown("### 🌟 주요 영향 변수 (Feature Importance TOP 10)"))

if {"feature", "importance"}.issubset(feature_importance.columns):
    plot_df = feature_importance.sort_values("importance", ascending=True).tail(10)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(plot_df["feature"], plot_df["importance"], color='teal')
    ax.set_title("Feature Importance TOP 10")
    plt.show()
else:
    print("⚠️ feature_importance.csv 형식이 맞지 않습니다.")

display(Markdown("### 🔍 고위험 예측 데이터 미리보기 (Top 10)"))
preview_cols = ["pred_prob", "risk_level", "returned", "product_price", "discount_percent", 
                "past_return_rate", "delay_days_int", "product_category_label"]
valid_cols = [c for c in preview_cols if c in filtered_df.columns]

display(filtered_df[valid_cols].sort_values("pred_prob", ascending=False).head(10))